<a href="https://colab.research.google.com/github/aman-verma02/Network-Security-System/blob/main/Colab-GoogleAutoencoder_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/aman-verma02/Network-Security-System.git
%cd Network-Security-System
!pip install -r requirements.txt

Mounted at /content/drive
Cloning into 'Network-Security-System'...
remote: Enumerating objects: 257, done.
remote: Counting objects: 100% (257/257), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 257 (delta 100), reused 197 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (257/257), 5.42 MiB | 6.22 MiB/s, done.
Resolving deltas: 100% (100/100), done.
/content/Network-Security-System
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.6/818.6 kB 51.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 77.7 M

In [3]:
import shutil

# Copy your 3 existing models into the cloned repo
shutil.copy('/content/drive/MyDrive/CICIDS_Dataset/model/model.pkl', 'final_model/model.pkl')
shutil.copy('/content/drive/MyDrive/CICIDS_Dataset/model/preprocessor.pkl', 'final_model/preprocessor.pkl')
shutil.copy('/content/drive/MyDrive/CICIDS_Dataset/model/feature_names.pkl', 'final_model/feature_names.pkl')

'final_model/feature_names.pkl'

In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
import pickle

# ── Step 1: Load cleaned CSV ──
df = pd.read_csv('/content/drive/MyDrive/CICIDS_Dataset/cicids_cleaned.csv')
print("Loaded:", df.shape)

# ── Step 2: Split features and target ──
X = df.drop(columns=['target'])
y = df['target']

# ── Step 3: Train/test split ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

# ── Step 4: Drop highly correlated features ──
corr_matrix = X_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]
X_train = X_train.drop(columns=drop_corr)
X_test = X_test.drop(columns=drop_corr)
print(f"After correlation drop: {X_train.shape}")

# ── Step 5: Drop low variance features ──
selector = VarianceThreshold(threshold=0.01)
selector.fit(X_train)
X_train = X_train[X_train.columns[selector.get_support()]]
X_test = X_test[X_test.columns[selector.get_support()]]
print(f"After variance threshold: {X_train.shape}")

# ── Step 6: Select top 40 features by RF importance ──
X_sample = X_train.sample(n=min(50000, len(X_train)), random_state=42)
y_sample = y_train[X_sample.index]

rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf.fit(X_sample, y_sample)

importances = pd.Series(rf.feature_importances_, index=X_train.columns)
top_features = importances.nlargest(40).index.tolist()
X_train = X_train[top_features]
X_test = X_test[top_features]
print(f"After feature selection: {X_train.shape}")

# ── Step 7: Scale ──
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ── Step 8: Build numpy arrays (features + target combined) ──
train_arr = np.c_[X_train_scaled, np.array(y_train)]
test_arr = np.c_[X_test_scaled, np.array(y_test)]
print("train_arr shape:", train_arr.shape)
print("test_arr shape:", test_arr.shape)

# ── Step 9: Save arrays and feature names to Drive ──
np.save('/content/drive/MyDrive/CICIDS_Dataset/train_arr.npy', train_arr)
np.save('/content/drive/MyDrive/CICIDS_Dataset/test_arr.npy', test_arr)

with open('/content/drive/MyDrive/CICIDS_Dataset/model/feature_names.pkl', 'wb') as f:
    pickle.dump(top_features, f)

with open('/content/drive/MyDrive/CICIDS_Dataset/model/preprocessor.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("All saved to Drive!")

Loaded: (836641, 101)
Train: (669312, 100) Test: (167329, 100)
After correlation drop: (669312, 62)
After variance threshold: (669312, 57)
After feature selection: (669312, 40)
train_arr shape: (669312, 41)
test_arr shape: (167329, 41)
All saved to Drive!


In [8]:
import numpy as np
from networksecurity.utils.main_utils.utils import load_numpy_array_data
from networksecurity.components.anomaly_detector import AnomalyDetector

# Load your transformed arrays from Drive
train_arr = np.load('/content/drive/MyDrive/CICIDS_Dataset/train_arr.npy')
test_arr = np.load('/content/drive/MyDrive/CICIDS_Dataset/test_arr.npy')

X_train, y_train = train_arr[:, :-1], train_arr[:, -1]
X_test, y_test = test_arr[:, :-1], test_arr[:, -1]

# Train autoencoder
anomaly_detector = AnomalyDetector(
    input_dim=X_train.shape[1],
    epochs=20,
    batch_size=256,
    lr=0.001
)

results = anomaly_detector.initiate_anomaly_detector(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    save_path='final_model/autoencoder.pkl'
)

print(results)

{'threshold': 1.0181893110275269, 'accuracy': 0.6129003340723963, 'detection_rate': 0.0425037037037037, 'false_alarm_rate': 0.0014224323593344619}


In [10]:
shutil.copy('final_model/autoencoder.pkl', '/content/drive/MyDrive/CICIDS_Dataset/model/autoencoder.pkl')
print("Done!")

Done!
